In [121]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [122]:
matches_df = pd.read_csv(r"C:\Users\jaija.JAILAYA\Desktop\Projects\datasets\matches.csv")
deliveries_df = pd.read_csv(r"C:\Users\jaija.JAILAYA\Desktop\Projects\datasets\deliveries.csv")

In [123]:
# Display first few rows of both datasets
print("Matches Dataset Head:\n", matches_df.head())

Matches Dataset Head:
        id   season        city        date match_type player_of_match  \
0  335982  2007/08   Bangalore  2008-04-18     League     BB McCullum   
1  335983  2007/08  Chandigarh  2008-04-19     League      MEK Hussey   
2  335984  2007/08       Delhi  2008-04-19     League     MF Maharoof   
3  335985  2007/08      Mumbai  2008-04-20     League      MV Boucher   
4  335986  2007/08     Kolkata  2008-04-20     League       DJ Hussey   

                                        venue                        team1  \
0                       M Chinnaswamy Stadium  Royal Challengers Bangalore   
1  Punjab Cricket Association Stadium, Mohali              Kings XI Punjab   
2                            Feroz Shah Kotla             Delhi Daredevils   
3                            Wankhede Stadium               Mumbai Indians   
4                                Eden Gardens        Kolkata Knight Riders   

                         team2                  toss_winner toss_deci

In [124]:
print("Deliveries Dataset Head:\n", deliveries_df.head())

Deliveries Dataset Head:
    match_id  inning           batting_team                 bowling_team  over  \
0    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
1    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
2    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
3    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
4    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   

   ball       batter   bowler  non_striker  batsman_runs  extra_runs  \
0     1   SC Ganguly  P Kumar  BB McCullum             0           1   
1     2  BB McCullum  P Kumar   SC Ganguly             0           0   
2     3  BB McCullum  P Kumar   SC Ganguly             0           1   
3     4  BB McCullum  P Kumar   SC Ganguly             0           0   
4     5  BB McCullum  P Kumar   SC Ganguly             0           0   

   total_runs extras_type  is_wicket player_dismissed 

In [125]:
print("Matches Dataset Null Values:\n", matches_df.isnull().sum())

Matches Dataset Null Values:
 id                    0
season                0
city                 51
date                  0
match_type            0
player_of_match       5
venue                 0
team1                 0
team2                 0
toss_winner           0
toss_decision         0
winner                5
result                0
result_margin        19
target_runs           3
target_overs          3
super_over            0
method             1074
umpire1               0
umpire2               0
dtype: int64


In [126]:
print("Deliveries Dataset Null Values:\n", deliveries_df.isnull().sum())

Deliveries Dataset Null Values:
 match_id                 0
inning                   0
batting_team             0
bowling_team             0
over                     0
ball                     0
batter                   0
bowler                   0
non_striker              0
batsman_runs             0
extra_runs               0
total_runs               0
extras_type         246795
is_wicket                0
player_dismissed    247970
dismissal_kind      247970
fielder             251566
dtype: int64


In [127]:
# Fill missing values in matches
matches_df.fillna({
    "result_margin": 0,
    "winner": "No Result"
}, inplace=True)


In [128]:
deliveries_df.fillna({
    "player_dismissed": "Not Out",
    "dismissal_kind": "Not Dismissed",
    "extras_type": "No Extras"
}, inplace=True)

In [129]:
# Standardizing team names
team_name_mapping = {
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings",
    "Deccan Chargers": "Sunrisers Hyderabad",
    "Rising Pune Supergiant": "Rising Pune Supergiants",
    "Rising Pune Supergiants": "Rising Pune Supergiant",
}

In [130]:
# Apply standardization
for col in ["team1", "team2", "toss_winner", "winner"]:
    matches_df[col] = matches_df[col].replace(team_name_mapping)

for col in ["batting_team", "bowling_team"]:
    deliveries_df[col] = deliveries_df[col].replace(team_name_mapping)

In [131]:
# Fill missing city values based on venue mapping
venue_city_mapping = {
    "Dubai International Cricket Stadium": "Dubai",
    "Sharjah Cricket Stadium": "Sharjah",
    "Sheikh Zayed Stadium": "Abu Dhabi",
}
matches_df["city"] = matches_df.apply(
    lambda row: venue_city_mapping.get(row["venue"], row["city"]), axis=1
)

In [132]:
# Rename team columns for clarity
matches_df.rename(columns={"team1": "home_team", "team2": "away_team"}, inplace=True)

In [133]:
# Drop unnecessary columns
unnecessary_columns = ["match_type", "player_of_match", "target_runs", "target_overs", 
                       "super_over", "method", "umpire1", "umpire2"]
matches_df.drop(columns=unnecessary_columns, inplace=True)

In [134]:
# Compute cumulative features in deliveries data
deliveries_df["cum_runs"] = deliveries_df.groupby(["match_id", "inning"])["total_runs"].cumsum()
deliveries_df["cum_wickets"] = deliveries_df.groupby(["match_id", "inning"])["is_wicket"].cumsum()


In [135]:
# Compute overs completed in decimal format
deliveries_df["overs_completed"] = deliveries_df["over"] + (deliveries_df["ball"] - 1) / 6

In [136]:
# Compute current run rate safely
deliveries_df["current_run_rate"] = deliveries_df["cum_runs"] / deliveries_df["overs_completed"]
deliveries_df["current_run_rate"] = deliveries_df["current_run_rate"].fillna(0)

In [137]:
# Extract target score from first innings
first_innings_scores = (
    deliveries_df[deliveries_df["inning"] == 1]
    .groupby("match_id")["cum_runs"]
    .max()
    .reset_index()
)
first_innings_scores.rename(columns={"cum_runs": "target_score"}, inplace=True)
first_innings_scores["target_score"] += 1  # Target is always one more than first innings score

In [138]:
# Merge target into deliveries
deliveries_df = deliveries_df.merge(first_innings_scores, on="match_id", how="left")

In [139]:
# Compute required run rate for second innings
deliveries_df["remaining_overs"] = 20 - deliveries_df["overs_completed"]

In [140]:
# Prevent division by zero
deliveries_df["required_run_rate"] = deliveries_df.apply(
    lambda row: (row["target_score"] - row["cum_runs"]) / row["remaining_overs"] 
    if row["remaining_overs"] > 0 else 0, axis=1
)

In [141]:
# Remove infinite values
deliveries_df["required_run_rate"] = deliveries_df["required_run_rate"].replace([float("inf"), -float("inf")], 0)
deliveries_df["required_run_rate"] = deliveries_df["required_run_rate"].fillna(0)

In [142]:
# Merge preprocessed matches data with deliveries data
merged_df = deliveries_df.merge(matches_df, left_on="match_id", right_on="id", how="left")
merged_df.drop(columns=["id"], inplace=True)

In [143]:
# Save the final merged dataset
output_file = r"C:\Users\jaija.JAILAYA\Desktop\Projects\datasets\processed_cricket_data.csv"
merged_df.to_csv(output_file, index=False)

In [144]:
print(f"Processed dataset saved to {output_file}")

Processed dataset saved to C:\Users\jaija.JAILAYA\Desktop\Projects\datasets\processed_cricket_data.csv


In [145]:
# Load the processed dataset
processed_file = r"C:\Users\jaija.JAILAYA\Desktop\Projects\datasets\processed_cricket_data.csv"
processed_df = pd.read_csv(processed_file, low_memory=False)

In [147]:
# Get dataset information
print("\nProcessed Dataset Info:")
print(processed_df.info())


Processed Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 260920 entries, 0 to 260919
Data columns (total 35 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   match_id           260920 non-null  int64  
 1   inning             260920 non-null  int64  
 2   batting_team       260920 non-null  object 
 3   bowling_team       260920 non-null  object 
 4   over               260920 non-null  int64  
 5   ball               260920 non-null  int64  
 6   batter             260920 non-null  object 
 7   bowler             260920 non-null  object 
 8   non_striker        260920 non-null  object 
 9   batsman_runs       260920 non-null  int64  
 10  extra_runs         260920 non-null  int64  
 11  total_runs         260920 non-null  int64  
 12  extras_type        260920 non-null  object 
 13  is_wicket          260920 non-null  int64  
 14  player_dismissed   260920 non-null  object 
 15  dismissal_kind     260920 

In [148]:
# Display the first few rows
print("Processed Dataset Head:\n", processed_df.head())

Processed Dataset Head:
    match_id  inning           batting_team                 bowling_team  over  \
0    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
1    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
2    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
3    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
4    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   

   ball       batter   bowler  non_striker  batsman_runs  ...       city  \
0     1   SC Ganguly  P Kumar  BB McCullum             0  ...  Bangalore   
1     2  BB McCullum  P Kumar   SC Ganguly             0  ...  Bangalore   
2     3  BB McCullum  P Kumar   SC Ganguly             0  ...  Bangalore   
3     4  BB McCullum  P Kumar   SC Ganguly             0  ...  Bangalore   
4     5  BB McCullum  P Kumar   SC Ganguly             0  ...  Bangalore   

         date                  

In [149]:
# Check for missing values
print("\nMissing Values in Processed Dataset:\n", processed_df.isnull().sum())


Missing Values in Processed Dataset:
 match_id                  0
inning                    0
batting_team              0
bowling_team              0
over                      0
ball                      0
batter                    0
bowler                    0
non_striker               0
batsman_runs              0
extra_runs                0
total_runs                0
extras_type               0
is_wicket                 0
player_dismissed          0
dismissal_kind            0
fielder              251566
cum_runs                  0
cum_wickets               0
overs_completed           0
current_run_rate          0
target_score              0
remaining_overs           0
required_run_rate         0
season                    0
city                      0
date                      0
venue                     0
home_team                 0
away_team                 0
toss_winner               0
toss_decision             0
winner                    0
result                    0
result_ma

In [150]:
# Get basic statistics
print("\nProcessed Dataset Summary Statistics:\n", processed_df.describe())


Processed Dataset Summary Statistics:
            match_id         inning           over           ball  \
count  2.609200e+05  260920.000000  260920.000000  260920.000000   
mean   9.070665e+05       1.483531       9.197677       3.624486   
std    3.679913e+05       0.502643       5.683484       1.814920   
min    3.359820e+05       1.000000       0.000000       1.000000   
25%    5.483340e+05       1.000000       4.000000       2.000000   
50%    9.809670e+05       1.000000       9.000000       4.000000   
75%    1.254066e+06       2.000000      14.000000       5.000000   
max    1.426312e+06       6.000000      19.000000      11.000000   

        batsman_runs     extra_runs     total_runs      is_wicket  \
count  260920.000000  260920.000000  260920.000000  260920.000000   
mean        1.265001       0.067806       1.332807       0.049632   
std         1.639298       0.343265       1.626416       0.217184   
min         0.000000       0.000000       0.000000       0.000000   
25

In [151]:
required_features = [
    "inning", "cum_runs", "cum_wickets", "current_run_rate", "required_run_rate", 
    "target_score", "batting_team", "bowling_team", "city", "winner"
]

missing_features = [feature for feature in required_features if feature not in processed_df.columns]

if missing_features:
    print("Missing Features:", missing_features)
else:
    print("All required features are present in the final dataset.")


All required features are present in the final dataset.


In [152]:
# Encode categorical columns
label_encoder = LabelEncoder()
processed_df["batting_team_encoded"] = label_encoder.fit_transform(processed_df["batting_team"])
processed_df["bowling_team_encoded"] = label_encoder.fit_transform(processed_df["bowling_team"])
processed_df["city_encoded"] = label_encoder.fit_transform(processed_df["city"])

In [153]:
# Create 'win' feature (1 if batting_team wins, else 0)
processed_df["win"] = (processed_df["batting_team"] == processed_df["winner"]).astype(int)

In [154]:
# Select final features for the model
final_features = [
    "inning", "cum_runs", "cum_wickets", "current_run_rate", "required_run_rate",
    "target_score", "batting_team_encoded", "bowling_team_encoded", "city_encoded", "win"
]

final_df = processed_df[final_features]

In [155]:
# Train-test split
X = final_df.drop(columns=["win"])
y = final_df["win"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

train_data = pd.concat([X_train, y_train], axis=1)
test_data = pd.concat([X_test, y_test], axis=1)

In [156]:
# Save train and test datasets
train_file = r"C:\Users\jaija.JAILAYA\Desktop\Projects\datasets\train_data.csv"
test_file = r"C:\Users\jaija.JAILAYA\Desktop\Projects\datasets\test_data.csv"

train_df = pd.read_csv(train_file)
test_df = pd.read_csv(test_file)

train_data.to_csv(train_file, index=False)
test_data.to_csv(test_file, index=False)

In [204]:
print(f"Train and test datasets saved to:\n{train_file}\n{test_file}")

Train and test datasets saved to:
C:\Users\jaija.JAILAYA\Desktop\Projects\datasets\train_data.csv
C:\Users\jaija.JAILAYA\Desktop\Projects\datasets\test_data.csv


In [206]:
# Verify train-test split
print(f"Train dataset shape: {train_df.shape}")
print(f"Test dataset shape: {test_df.shape}")


Train dataset shape: (208736, 10)
Test dataset shape: (52184, 10)


In [208]:
print("\nTrain Dataset Sample:\n", train_df.head())
print("\nTest Dataset Sample:\n", test_df.head())


Train Dataset Sample:
    inning  cum_runs  cum_wickets  current_run_rate  required_run_rate  \
0       2        13            0          8.666667           7.351351   
1       1         0            0          0.000000           7.900000   
2       1       112            5          9.739130           6.941176   
3       2        53            1          7.395349           8.337662   
4       2       126            2          8.790698           8.823529   

   target_score  batting_team_encoded  bowling_team_encoded  city_encoded  win  
0           149                     1                     5            10    0  
1           158                    15                     7            16    0  
2           171                    15                     9             9    1  
3           160                     7                    11            30    1  
4           176                     9                    15             7    1  

Test Dataset Sample:
    inning  cum_runs  cum_wic

In [211]:
# Check distribution of 'win'
print("\nWin distribution in Train Set:")
print(train_df["win"].value_counts(normalize=True))

print("\nWin distribution in Test Set:")
print(test_df["win"].value_counts(normalize=True))


Win distribution in Train Set:
win
0    0.511402
1    0.488598
Name: proportion, dtype: float64

Win distribution in Test Set:
win
0    0.511402
1    0.488598
Name: proportion, dtype: float64
